In [1]:
from dotenv import load_dotenv
import os
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [2]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.agents.middleware import PIIMiddleware
 ##dummy tool
@tool
def customer_lookuo(query:str):
    """look up customer information"""
    return f"customer record found for query{query}"

agent = create_agent(
    model = "groq:qwen/qwen3.8-27b",
    tools=[customer_lookuo],
    middleware = [
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True
        ),
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True
        ),
        PIIMiddleware(
            "api_key",
            detector = "sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input = True
        )
    ]
)
print("agent with PII middleware is ready")


agent with PII middleware is ready


In [4]:
res = agent.invoke({
    "messages":[{
        "role":"user",
        "content":"my email is pramod4342@gmail.com and my credit card number is 4242-4242-4242-4242. can you help me?"
    }]
})
print("=====agent response=====")
print(res["messages"][-1].content)

=====agent response=====
I’m here to help, but I can’t access or verify sensitive personal information like your full email or credit card number, and you should be careful about sharing those in chat.

If you’d like, tell me:

1. What you’re trying to accomplish (e.g., update billing info, check an order, troubleshoot an issue).
2. Any non-sensitive details that might help (e.g., “I have a subscription,” “I got an error when paying,” “my order number is #12345”).

I can guide you through the steps or look up general customer information if you provide a safe identifier (like a last name or order ID, depending on what’s available).


In [5]:
res

{'messages': [HumanMessage(content='my email is [REDACTED_EMAIL] and my credit card number is ****-****-****-4242. can you help me?', additional_kwargs={}, response_metadata={}, id='6f0eaf46-72e9-4e19-a6ea-6ee698f44e50'),
  AIMessage(content='I’m here to help, but I can’t access or verify sensitive personal information like your full email or credit card number, and you should be careful about sharing those in chat.\n\nIf you’d like, tell me:\n\n1. What you’re trying to accomplish (e.g., update billing info, check an order, troubleshoot an issue).\n2. Any non-sensitive details that might help (e.g., “I have a subscription,” “I got an error when paying,” “my order number is #12345”).\n\nI can guide you through the steps or look up general customer information if you provide a safe identifier (like a last name or order ID, depending on what’s available).', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 147, 'prompt_tokens': 302, 'total_tokens': 449, 'comple